# US Traffic Accidents — подготовка и анализ данных

Этот ноутбук автоматически получает данные и подготавливает их для анализа. Это первый этап проекта: SQL, EDA, сравнение инструментов и модели будут добавлены далее. Запускайте ячейки сверху вниз.

In [ ]:
%pip install -q pandas pyarrow plotly matplotlib scipy polars psutil scikit-learn joblib

## 1. Автоматическая загрузка данных

Публичная копия файла March 2023 скачивается в `/content`. Её SHA-256 совпадает с архивом, предоставленным автором проекта. Загрузка составляет 3,06 ГБ и требует интернета и свободного места. Файлы в `/content` удаляются при завершении сеанса Colab. Источник: [Hugging Face](https://huggingface.co/datasets/Sanjana7787/us-accidents-2016-2023), исходный набор [Kaggle](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents).

In [ ]:
from pathlib import Path
import hashlib
import urllib.request

DATA_PATH = Path("/content/US_Accidents_March23.csv")
URL = "https://huggingface.co/datasets/Sanjana7787/us-accidents-2016-2023/resolve/b514f1c/US_Accidents_March23.csv"
EXPECTED_SHA256 = "e3e9f962e3e2289db1bdce91623abbcf9ba4d7bfc1adb091c8aeee6c2639d204"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download_dataset():
    if DATA_PATH.exists():
        if sha256_file(DATA_PATH) == EXPECTED_SHA256:
            print("Проверенный файл уже загружен.")
            return
        raise ValueError(f"Контрольная сумма {DATA_PATH} не совпадает. Укажите другой путь или удалите повреждённый файл вручную.")
    temp = DATA_PATH.with_suffix(".download")
    try:
        print("Загружается файл 3,06 ГБ; это может занять время...")
        with urllib.request.urlopen(URL, timeout=60) as response, temp.open("wb") as out:
            while True:
                block = response.read(8 * 1024 * 1024)
                if not block:
                    break
                out.write(block)
        if sha256_file(temp) != EXPECTED_SHA256:
            raise ValueError("Контрольная сумма загруженного файла не совпадает.")
        temp.replace(DATA_PATH)
        print("Данные загружены и проверены.")
    except Exception:
        temp.unlink(missing_ok=True)
        raise

download_dataset()

## 2. Настройки

`MAX_ROWS = None` означает обработку всего набора. Для короткой проверки можно указать `10_000`, но нужно использовать другую папку результатов.

Результаты сохраняются в `/content` на время текущего сеанса Colab. Если папка уже содержит результаты подходящего запуска, ноутбук использует их повторно. После завершения сеанса Colab эти файлы могут исчезнуть; тогда загрузку и обработку придётся выполнить снова. Для постоянного хранения позже добавим отдельный вариант с Google Drive.

In [ ]:
ARCHIVE_PATH = DATA_PATH
OUTPUT_DIR = "/content/traffic_project/full_parquet"
CHUNK_SIZE = 100_000
MAX_ROWS = None  # None означает все строки

## 3. Обработка

Пропуски и Severity подсчитываются до преобразований; некорректные даты и метки фиксируются, но строки не удаляются. Память до и после — сумма размеров чанков, а не пиковая память процесса. Год 0 означает неразобранную или отсутствующую дату.

In [ ]:
"""Chunked conversion of the US Accidents ZIP to year-partitioned Parquet."""
import json
import time
import zipfile
from contextlib import contextmanager
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


@contextmanager
def open_dataset(path):
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            members = [n for n in archive.namelist() if n.lower().endswith('.csv')]
            if len(members) != 1:
                raise ValueError('The archive must contain exactly one CSV file')
            with archive.open(members[0]) as source:
                yield source
    else:
        with path.open('rb') as source:
            yield source


def prepare_data(archive_path, output_dir, chunk_size=100_000, max_rows=None):
    archive_path, output_dir = Path(archive_path), Path(output_dir)
    if chunk_size < 1 or (max_rows is not None and max_rows < 1):
        raise ValueError('chunk_size and max_rows must be positive')
    # Refuse to mix a previous run with the current dataset.
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError(f'Output directory is not empty: {output_dir}')
    output_dir.mkdir(parents=True, exist_ok=True)
    totals = {'rows': 0, 'missing': {}, 'severity': {}, 'invalid_start_time': 0,
              'invalid_severity': 0, 'memory_before_bytes': 0, 'memory_after_bytes': 0}
    started = time.perf_counter()
    with open_dataset(archive_path) as source:
        reader = pd.read_csv(source, chunksize=chunk_size, dtype=str, keep_default_na=True,
                             nrows=max_rows, low_memory=False)
        for number, chunk in enumerate(reader):
            totals['rows'] += len(chunk)
            totals['memory_before_bytes'] += int(chunk.memory_usage(deep=True).sum())
            for col, value in chunk.isna().sum().items():
                totals['missing'][col] = totals['missing'].get(col, 0) + int(value)
            for value, count in chunk['Severity'].fillna('<missing>').value_counts().items():
                totals['severity'][value] = totals['severity'].get(value, 0) + int(count)
            dates = pd.to_datetime(chunk['Start_Time'], errors='coerce', format='mixed')
            # Older Pandas versions do not support format='mixed'.
            if dates.isna().all() and chunk['Start_Time'].notna().any():
                dates = pd.to_datetime(chunk['Start_Time'], errors='coerce')
            totals['invalid_start_time'] += int((dates.isna() & chunk['Start_Time'].notna()).sum())
            totals['invalid_severity'] += int((~chunk['Severity'].isin(['1', '2', '3', '4']) & chunk['Severity'].notna()).sum())
            for col in ['Severity', 'State', 'Sunrise_Sunset', 'Weather_Condition']:
                chunk[col] = chunk[col].astype('category')
            totals['memory_after_bytes'] += int(chunk.memory_usage(deep=True).sum())
            years = dates.dt.year.fillna(0).astype(int)
            for year in sorted(years.unique()):
                target = output_dir / f'year={year}'
                target.mkdir(exist_ok=True)
                part = chunk.loc[years == year].copy()
                # Strings ensure a stable schema across chunks and partitions.
                for col in part.columns:
                    part[col] = part[col].astype('string')
                table = pa.Table.from_pandas(part, preserve_index=False)
                pq.write_table(table, target / f'part-{number:05d}.parquet', compression='snappy')
            print(f'Processed {totals["rows"]:,} rows', flush=True)
    totals['elapsed_seconds'] = round(time.perf_counter() - started, 2)
    totals['scope'] = 'full archive' if max_rows is None else f'first {max_rows} rows at most'
    totals['memory_note'] = 'Sum of per-chunk dataframe memory, not peak process memory.'
    (output_dir / 'audit.json').write_text(json.dumps(totals, indent=2, ensure_ascii=False), encoding='utf-8')
    return totals


In [ ]:
# Если обработка уже завершена, читаем готовый отчёт.
# Это позволяет повторно запускать графики в текущем сеансе без обработки CSV.
audit_file = Path(OUTPUT_DIR) / "audit.json"
expected_scope = "full archive" if MAX_ROWS is None else f"first {MAX_ROWS} rows at most"

if audit_file.exists():
    with open(audit_file, encoding="utf-8") as file:
        audit = json.load(file)
    if audit["scope"] != expected_scope:
        raise ValueError("В этой папке результаты другого режима. Выберите другую папку OUTPUT_DIR.")
    print("Используем результаты предыдущей обработки.")
else:
    audit = prepare_data(ARCHIVE_PATH, OUTPUT_DIR, CHUNK_SIZE, MAX_ROWS)

print(json.dumps(audit, ensure_ascii=False, indent=2))

## 4. Проверка

Проверим, что количество строк в файлах Parquet совпадает с отчётом. Если проверка не прошла, создайте новую папку в настройках и выполните обработку снова.

In [ ]:
files = list(Path(OUTPUT_DIR).glob("year=*/part-*.parquet"))
parquet_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
print("Parquet files:", len(files), "rows:", parquet_rows)
assert parquet_rows == audit["rows"]

## 5. Простой анализ данных

Посчитаем число записей по тяжести, штатам, часам, годам и погодным условиям. Читаем файлы по очереди, чтобы экономить память. Это количество зарегистрированных ДТП, а не риск ДТП: данные о количестве поездок и населении здесь не учитываются.

In [ ]:
"""Simple analysis of Parquet files, one file at a time."""
from pathlib import Path

import pandas as pd


def add_counts(total, current):
    """Add counts from one file to the counts from previous files."""
    for label, count in current.items():
        total[label] = total.get(label, 0) + int(count)


def analyze_files(folder):
    files = sorted(Path(folder).glob('year=*/part-*.parquet'))
    if not files:
        raise FileNotFoundError(f'No Parquet files found in {folder}')

    counts = {'severity': {}, 'state': {}, 'hour': {}, 'weather': {}, 'year': {}}
    columns = ['Severity', 'State', 'Start_Time', 'Weather_Condition']
    for file in files:
        data = pd.read_parquet(file, columns=columns)
        add_counts(counts['severity'], data['Severity'].dropna().value_counts())
        add_counts(counts['state'], data['State'].dropna().value_counts())
        add_counts(counts['weather'], data['Weather_Condition'].dropna().value_counts())
        # The hours and years are in fixed positions in the original date string.
        add_counts(counts['hour'], data['Start_Time'].str.slice(11, 13).dropna().value_counts())
        add_counts(counts['year'], data['Start_Time'].str.slice(0, 4).dropna().value_counts())

    results = {}
    for name, values in counts.items():
        table = pd.DataFrame(list(values.items()), columns=[name, 'accidents'])
        results[name] = table.sort_values('accidents', ascending=False).reset_index(drop=True)
    return results


In [ ]:
results = analyze_files(OUTPUT_DIR)

# Сохраним таблицы для дальнейшей работы.
results_dir = Path("/content/traffic_project/tables")
results_dir.mkdir(parents=True, exist_ok=True)
for name, table in results.items():
    table.to_csv(results_dir / f"{name}.csv", index=False)

print("Данные обработаны. Пример таблицы:")
display(results["severity"])

## 6. Графики

Все графики интерактивные: можно навести курсор на столбец и увидеть число записей. Severity показывает влияние ДТП на движение от 1 до 4, а не степень травм.

In [ ]:
import plotly.express as px

severity = results["severity"].sort_values("severity")
px.bar(severity, color_discrete_sequence=["#507dbc"], x="severity", y="accidents", title="Количество ДТП по уровню влияния на движение", labels={"severity": "Severity", "accidents": "Количество ДТП"}).show()

In [ ]:
states = results["state"].head(10)
px.bar(states, color_discrete_sequence=["#2a9d8f"], x="state", y="accidents", title="10 штатов с наибольшим числом записей", labels={"state": "Штат", "accidents": "Количество ДТП"}).show()

In [ ]:
hours = results["hour"].sort_values("hour")
px.bar(hours, color_discrete_sequence=["#e9a23b"], x="hour", y="accidents", title="Распределение ДТП по часу начала", labels={"hour": "Местный час", "accidents": "Количество ДТП"}).show()

In [ ]:
years = results["year"].sort_values("year")
px.bar(years, color_discrete_sequence=["#8064a2"], x="year", y="accidents", title="Количество записей по годам", labels={"year": "Год", "accidents": "Количество ДТП"}).show()

In [ ]:
weather = results["weather"].head(10)
px.bar(weather, color_discrete_sequence=["#d96c6c"], x="weather", y="accidents", title="10 наиболее частых погодных условий", labels={"weather": "Погодные условия", "accidents": "Количество ДТП"}).show()

Число записей зависит от покрытия источников, размера штата и других факторов. В 2023 году данные заканчиваются в марте, поэтому сравнивать его с полными годами напрямую нельзя. Пропущенная погода в график погодных условий не входит. Следующие этапы: аудит качества, SQL, проверка гипотез, сравнение инструментов и модели.

## 7. База данных SQLite

SQLite — это база данных, которая хранится в одном файле. Она входит в стандартную библиотеку Python, поэтому устанавливать отдельный сервер не нужно.

Мы создадим две связанные таблицы:

- `states` — справочник штатов: номер `state_id` и обозначение штата `state_code`.
- `accidents` — записи о ДТП: собственный номер, ссылка `state_id`, `severity`, время начала и погода.

**ER-диаграмма** (Entity–Relationship) показывает связь таблиц:

```text
states                              accidents
-------------------                 -------------------------
state_id (PK)  1 ────────────  ∞   state_id (FK)
state_code                          accident_id (PK)
                                    severity
                                    start_time
                                    weather
```

PK — первичный ключ, который отличает строки таблицы. FK — внешний ключ: он связывает ДТП со штатом. Один штат может иметь много записей ДТП. Если штат неизвестен, `state_id` остаётся пустым.

Мы переносим в SQLite только столбцы, нужные для этих запросов; полный набор остаётся в Parquet. Первая загрузка базы может занять несколько минут и дополнительное место.

In [ ]:
"""SQLite: загрузка записей ДТП и аналитические запросы."""
import sqlite3
from pathlib import Path
import pandas as pd


def create_database(parquet_folder, database_path):
    """Создаёт базу и загружает данные из Parquet по одному файлу."""
    files = sorted(Path(parquet_folder).glob('year=*/part-*.parquet'))
    if not files:
        raise FileNotFoundError('Файлы Parquet не найдены')
    database_path = Path(database_path)
    if database_path.exists():
        raise FileExistsError(f'База уже существует: {database_path}')
    database_path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(database_path) as connection:
        connection.execute('PRAGMA foreign_keys = ON')
        connection.executescript('''
            CREATE TABLE states (
                state_id INTEGER PRIMARY KEY,
                state_code TEXT NOT NULL UNIQUE
            );
            CREATE TABLE accidents (
                accident_id INTEGER PRIMARY KEY,
                state_id INTEGER,
                severity INTEGER,
                start_time TEXT,
                weather TEXT,
                FOREIGN KEY (state_id) REFERENCES states(state_id)
            );
            CREATE TABLE project_info (row_count INTEGER NOT NULL);
        ''')
        # Справочник штатов избегает многократного хранения названия штата.
        state_ids = {}
        columns = ['State', 'Severity', 'Start_Time', 'Weather_Condition']
        for number, file in enumerate(files, 1):
            frame = pd.read_parquet(file, columns=columns)
            for state in frame['State'].dropna().unique():
                if state not in state_ids:
                    cursor = connection.execute('INSERT INTO states (state_code) VALUES (?)', (state,))
                    state_ids[state] = cursor.lastrowid
            state_column = frame['State'].map(state_ids)
            severity = pd.to_numeric(frame['Severity'], errors='coerce')
            rows = zip(state_column, severity, frame['Start_Time'], frame['Weather_Condition'])
            # Превращаем пропуски в None, чтобы SQLite сохранил их как NULL.
            batch = []
            for state, level, start, weather in rows:
                batch.append((None if pd.isna(state) else int(state),
                              None if pd.isna(level) else int(level),
                              None if pd.isna(start) else str(start),
                              None if pd.isna(weather) else str(weather)))
            connection.executemany(
                'INSERT INTO accidents (state_id, severity, start_time, weather) VALUES (?, ?, ?, ?)', batch)
            connection.commit()
            if number % 20 == 0 or number == len(files):
                print(f'Загружено файлов: {number} из {len(files)}')
        connection.execute('CREATE INDEX idx_accidents_state ON accidents(state_id)')
        connection.execute('INSERT INTO project_info SELECT COUNT(*) FROM accidents')
    return database_path


# Ключи описывают смысл запросов. Запросы используют одинаковые исходные данные.
QUERIES = {
    '1_severity': '''
        SELECT severity, COUNT(*) AS accidents
        FROM accidents GROUP BY severity ORDER BY severity
    ''',
    '2_states_join': '''
        SELECT s.state_code, COUNT(*) AS accidents
        FROM accidents a JOIN states s ON a.state_id = s.state_id
        GROUP BY s.state_code ORDER BY accidents DESC LIMIT 10
    ''',
    '3_states_having': '''
        SELECT s.state_code, COUNT(*) AS accidents,
               ROUND(AVG(a.severity), 2) AS average_severity
        FROM accidents a JOIN states s ON a.state_id = s.state_id
        GROUP BY s.state_code HAVING COUNT(*) >= 10000
        ORDER BY average_severity DESC
    ''',
    '4_hour': '''
        SELECT SUBSTR(start_time, 12, 2) AS hour, COUNT(*) AS accidents
        FROM accidents WHERE start_time IS NOT NULL
        GROUP BY hour ORDER BY hour
    ''',
    '5_weather': '''
        SELECT weather, COUNT(*) AS accidents
        FROM accidents WHERE weather IS NOT NULL
        GROUP BY weather ORDER BY accidents DESC LIMIT 10
    ''',
    '6_state_above_average': '''
        WITH state_counts AS (
            SELECT s.state_code, COUNT(*) AS accidents
            FROM accidents a JOIN states s ON a.state_id = s.state_id
            GROUP BY s.state_code
        )
        SELECT state_code, accidents FROM state_counts
        WHERE accidents > (SELECT AVG(accidents) FROM state_counts)
        ORDER BY accidents DESC
    ''',
    '7_state_rank': '''
        WITH state_counts AS (
            SELECT s.state_code, COUNT(*) AS accidents
            FROM accidents a JOIN states s ON a.state_id = s.state_id
            GROUP BY s.state_code
        )
        SELECT state_code, accidents,
               RANK() OVER (ORDER BY accidents DESC) AS state_rank
        FROM state_counts ORDER BY state_rank
    ''',
    '8_year_severity': '''
        SELECT SUBSTR(start_time, 1, 4) AS year, severity, COUNT(*) AS accidents
        FROM accidents WHERE start_time IS NOT NULL
        GROUP BY year, severity ORDER BY year, severity
    '''
}


def run_queries(database_path, output_folder):
    """Выполняет запросы, возвращает таблицы Pandas и сохраняет их в CSV."""
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    results = {}
    with sqlite3.connect(database_path) as connection:
        for name, query in QUERIES.items():
            table = pd.read_sql_query(query, connection)
            table.to_csv(output_folder / f'{name}.csv', index=False)
            results[name] = table
    return results


In [ ]:
# База хранится в /content до завершения сеанса Colab.
# Если база уже есть, проверяем, что предыдущая загрузка завершилась.
DATABASE_PATH = Path("/content/traffic_project/accidents.db")
if DATABASE_PATH.exists():
    try:
        with sqlite3.connect(DATABASE_PATH) as connection:
            saved_rows = connection.execute("SELECT row_count FROM project_info").fetchone()[0]
        if saved_rows != audit["rows"]:
            raise ValueError("Число строк в базе не совпадает с текущим набором.")
        print("Используем уже созданную базу:", saved_rows, "строк")
    except (sqlite3.Error, TypeError) as error:
        raise ValueError("Предыдущая загрузка базы не завершена. Выберите новый DATABASE_PATH.") from error
else:
    create_database(OUTPUT_DIR, DATABASE_PATH)
    print("База создана:", DATABASE_PATH)

## 8. Восемь SQL-запросов

В словаре `QUERIES` выше находятся все запросы. Функция `run_queries` выполняет каждый запрос, переводит его результат в Pandas DataFrame и сохраняет таблицу в CSV.

1. **Severity:** сколько записей относится к каждому уровню.
2. **JOIN:** соединяем ДТП со справочником штатов и выбираем 10 штатов с наибольшим числом записей.
3. **GROUP BY / HAVING:** группируем по штатам и оставляем только штаты с 10 000 или более записей. Среднее Severity здесь лишь описательный показатель для порядковой шкалы.
4. **Часы:** считаем количество ДТП по часу начала.
5. **Погода:** показываем 10 самых частых непустых значений.
6. **CTE и подзапрос:** сравниваем число записей штата со средним числом записей по штатам.
7. **Оконная функция RANK:** присваиваем штатам места по числу записей; при равенстве значений места совпадают.
8. **Годы и Severity:** считаем записи для каждой комбинации года и уровня.

Количество записей отражает доступные данные, а не риск ДТП. При небольшом демонстрационном наборе запрос №3 может вернуть пустую таблицу — это нормально.

In [ ]:
sql_results = run_queries(DATABASE_PATH, "/content/traffic_project/sql_results")
for name, table in sql_results.items():
    print("\nЗапрос:", name, "| строк:", len(table))
    display(table.head(10))

## 9. Проверка результатов SQL

Сравним число записей каждого уровня Severity из SQLite и из предыдущего анализа Parquet. Они должны совпадать. Это помогает заметить ошибку при переносе данных.

In [ ]:
parquet_counts = results["severity"].copy()
parquet_counts["severity"] = pd.to_numeric(parquet_counts["severity"], errors="coerce")
comparison = parquet_counts.merge(sql_results["1_severity"], on="severity", suffixes=("_parquet", "_sql"))
display(comparison)
assert len(comparison) == len(parquet_counts) == len(sql_results["1_severity"])
assert (comparison["accidents_parquet"] == comparison["accidents_sql"]).all()
print("Проверка пройдена: числа совпадают.")

## 10. Проверка качества данных

Прежде чем обучать модели, важно понять, насколько данные полные и корректные. Здесь проверяем:

- пропуски — по ним отчёт уже создан в переменной `audit`;
- повторяющиеся **ID**: считаем повторные строки с тем же ID, включая повторы между файлами; это не проверка всех возможных дублей ДТП с разными ID;
- даты начала и допустимые значения Severity от 1 до 4;
- температуру: непустые нечисловые значения или значения вне диапазона от −100 до 140 °F отмечаем как подозрительные. Это выбранное правило проверки, а не доказательство ошибки;
- выбросы температуры по методу IQR. Выброс — необычное значение, которое может быть настоящим; автоматически не удаляем его.

Для поиска повторов используем временную SQLite-таблицу на диске. Для расчёта IQR собираем числовые значения температуры в памяти, поэтому дополнительная память всё же понадобится. Первый запуск может занять несколько минут. Результаты сохраняются на время сеанса Colab.

In [ ]:
"""Проверка качества и две гипотезы на данных Parquet."""
from pathlib import Path
import sqlite3
import tempfile

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency


def weather_group(value):
    """Упрощённые группы описаний погоды; порядок условий важен."""
    if pd.isna(value):
        return 'Unknown'
    text = str(value).lower()
    if any(word in text for word in ['snow', 'sleet', 'ice', 'freezing', 'hail']):
        return 'Snow / ice'
    if any(word in text for word in ['rain', 'drizzle', 'thunder', 'shower']):
        return 'Rain / storm'
    if any(word in text for word in ['clear', 'fair', 'sunny']):
        return 'Clear / fair'
    if any(word in text for word in ['cloud', 'overcast']):
        return 'Cloudy'
    return 'Other'


def analyze_quality(folder):
    """Читает файлы по очереди и возвращает показатели качества и таблицы гипотез."""
    files = sorted(Path(folder).glob('year=*/part-*.parquet'))
    if not files:
        raise FileNotFoundError('Файлы Parquet не найдены')
    columns = ['ID', 'Severity', 'Start_Time', 'Temperature(F)', 'Weather_Condition', 'Sunrise_Sunset']
    quality = {'rows': 0, 'missing_id': 0, 'duplicate_id_rows': 0,
               'invalid_severity': 0, 'invalid_date': 0,
               'invalid_temperature': 0, 'missing_temperature': 0}
    temperatures = []
    weather_table = pd.DataFrame(0, index=['Clear / fair', 'Cloudy', 'Rain / storm', 'Snow / ice', 'Other'], columns=[1, 2, 3, 4], dtype='int64')
    day_table = pd.DataFrame(0, index=['Day', 'Night'], columns=[1, 2, 3, 4], dtype='int64')

    # Временная SQLite-таблица хранит уже встречавшиеся ID на диске.
    # Так мы считаем повторы между разными файлами, а не только внутри одного чанка.
    with tempfile.TemporaryDirectory() as temp_dir:
        with sqlite3.connect(Path(temp_dir) / 'ids.db') as db:
            db.execute('CREATE TABLE seen_ids (id TEXT PRIMARY KEY) WITHOUT ROWID')
            for index, file in enumerate(files, 1):
                frame = pd.read_parquet(file, columns=columns)
                quality['rows'] += len(frame)
                ids = frame['ID'].dropna().astype(str)
                quality['missing_id'] += int(frame['ID'].isna().sum())
                before = db.total_changes
                db.executemany('INSERT OR IGNORE INTO seen_ids (id) VALUES (?)', ((value,) for value in ids))
                added = db.total_changes - before
                quality['duplicate_id_rows'] += len(ids) - added
                db.commit()

                severity = pd.to_numeric(frame['Severity'], errors='coerce')
                valid_severity = severity.isin([1, 2, 3, 4])
                quality['invalid_severity'] += int((frame['Severity'].notna() & ~valid_severity).sum())
                date = pd.to_datetime(frame['Start_Time'], format='mixed', errors='coerce')
                quality['invalid_date'] += int((frame['Start_Time'].notna() & date.isna()).sum())
                temperature = pd.to_numeric(frame['Temperature(F)'], errors='coerce')
                quality['missing_temperature'] += int(frame['Temperature(F)'].isna().sum())
                # Границы являются правилом проверки, а не доказательством ошибки.
                invalid_temp = frame['Temperature(F)'].notna() & (temperature.isna() | ~temperature.between(-100, 140))
                quality['invalid_temperature'] += int(invalid_temp.sum())
                temperatures.append(temperature[temperature.between(-100, 140)].to_numpy(dtype='float32'))

                groups = frame['Weather_Condition'].map(weather_group)
                t1 = pd.crosstab(groups[valid_severity], severity[valid_severity]).reindex(index=weather_table.index, columns=weather_table.columns, fill_value=0)
                weather_table += t1
                days = frame['Sunrise_Sunset']
                t2 = pd.crosstab(days[valid_severity], severity[valid_severity]).reindex(index=day_table.index, columns=day_table.columns, fill_value=0)
                day_table += t2
                if index % 20 == 0 or index == len(files):
                    print(f'Проверено файлов: {index} из {len(files)}')

    values = np.concatenate(temperatures)
    if len(values):
        q1, q3 = np.percentile(values, [25, 75])
        lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
        quality['temperature_q1'] = float(q1)
        quality['temperature_q3'] = float(q3)
        quality['temperature_iqr_lower'] = float(lower)
        quality['temperature_iqr_upper'] = float(upper)
        quality['temperature_iqr_outliers'] = int(((values < lower) | (values > upper)).sum())
    return quality, weather_table, day_table


def test_hypothesis(table):
    """Критерий chi-square и размер эффекта Cramer's V."""
    table = table.loc[table.sum(axis=1) > 0, table.sum(axis=0) > 0]
    if min(table.shape) < 2:
        raise ValueError('Для проверки нужны как минимум две группы и два уровня Severity')
    chi2, p_value, degrees, expected = chi2_contingency(table)
    sample_size = table.to_numpy().sum()
    effect = np.sqrt(chi2 / (sample_size * min(table.shape[0] - 1, table.shape[1] - 1)))
    return {'observations': int(sample_size), 'chi2': float(chi2), 'p_value': float(p_value),
            'degrees_of_freedom': int(degrees), 'cramers_v': float(effect),
            'expected_below_5': int((expected < 5).sum())}


In [ ]:
# Если результаты уже созданы в текущем сеансе, используем их повторно.
quality_dir = Path("/content/traffic_project/quality")
quality_dir.mkdir(parents=True, exist_ok=True)
quality_file = quality_dir / "quality.json"
if quality_file.exists():
    with open(quality_file, encoding="utf-8") as file:
        quality = json.load(file)
    if quality["rows"] != audit["rows"]:
        raise ValueError("Аудит относится к другому числу строк. Выберите другую папку quality_dir.")
    weather_table = pd.read_csv(quality_dir / "weather.csv", index_col=0)
    day_table = pd.read_csv(quality_dir / "day.csv", index_col=0)
    weather_table.columns = weather_table.columns.astype(int)
    day_table.columns = day_table.columns.astype(int)
    print("Используем сохранённые результаты проверки качества.")
else:
    quality, weather_table, day_table = analyze_quality(OUTPUT_DIR)
    weather_table.to_csv(quality_dir / "weather.csv")
    day_table.to_csv(quality_dir / "day.csv")
    with open(quality_file, "w", encoding="utf-8") as file:
        json.dump(quality, file, ensure_ascii=False, indent=2)

print("Результаты проверки:")
display(pd.DataFrame(list(quality.items()), columns=["Показатель", "Значение"]))

### Пропуски и обычный график

Покажем десять столбцов, в которых пропущено больше всего значений. Количество пропусков посчитано по всем строкам выбранного режима.

In [ ]:
import matplotlib.pyplot as plt

missing = pd.DataFrame(list(audit["missing"].items()), columns=["Признак", "Пропуски"])
missing = missing.sort_values("Пропуски", ascending=False).head(10)
plt.figure(figsize=(10, 5))
plt.barh(missing["Признак"], missing["Пропуски"], color="#507dbc")
plt.gca().invert_yaxis()
plt.title("10 признаков с наибольшим числом пропусков")
plt.xlabel("Количество пропущенных значений")
plt.tight_layout()
plt.show()

## 11. Две гипотезы

**Гипотеза 1:** распределение уровней Severity связано с погодной группой.

**Гипотеза 2:** распределение Severity различается днём и ночью.

Для каждой гипотезы нулевая гипотеза (H0) утверждает, что связи нет. Критерий **chi-square** сравнивает наблюдаемые количества с ожидаемыми при H0. Если `p-value < 0.05`, мы отвергаем H0; это не доказывает причину. При миллионах записей даже слабая связь может дать очень маленькое p-value, поэтому дополнительно смотрим **Cramer's V**: чем ближе к нулю, тем слабее связь.

Неизвестная погода или время суток не входят в соответствующий тест; число использованных строк выводится. Погоду объединяем по словам в пять групп, поэтому классификация упрощённая и некоторые случаи попадают в Other. Сначала проверяется снег/лёд, затем дождь/гроза, ясная и облачная погода. Результаты зависят от этих правил. Записи могут быть зависимы, а состав и полнота данных различаются по штатам и времени. Это ограничивает интерпретацию p-value.

In [ ]:
print("Количество ДТП по погодным группам и Severity:")
display(weather_table)
print("Количество ДТП днём и ночью по Severity:")
display(day_table)

weather_test = test_hypothesis(weather_table)
day_test = test_hypothesis(day_table)
for name, result in [("Погода и Severity", weather_test), ("День/ночь и Severity", day_test)]:
    print("\n", name)
    print("Число записей:", result["observations"])
    print("chi-square:", round(result["chi2"], 2), "; степени свободы:", result["degrees_of_freedom"])
    # Очень маленькое число может отображаться как 0.0 из-за ограничений компьютера.
    print("p-value:", "< 1e-300 (меньше точности вычисления)" if result["p_value"] == 0 else result["p_value"])
    print("Cramer's V:", round(result["cramers_v"], 4))
    print("Ожидаемых ячеек меньше 5:", result["expected_below_5"])
    if result["expected_below_5"]:
        print("Внимание: малые ожидаемые частоты могут нарушать условия теста.")

### Второй обычный график: день и ночь

Сравниваем **доли** уровней Severity, а не абсолютные количества: записей днём и ночью может быть разное число.

In [ ]:
day_percent = day_table.div(day_table.sum(axis=1).replace(0, np.nan), axis=0) * 100
ax = day_percent.plot(kind="bar", stacked=True, figsize=(8, 5),
                      color=["#507dbc", "#2a9d8f", "#e9a23b", "#d96c6c"])
ax.set_title("Доли уровней Severity: день и ночь")
ax.set_xlabel("Время суток")
ax.set_ylabel("Доля записей, %")
ax.legend(title="Severity")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 12. Сравнение Pandas и Polars

Pandas и Polars — две библиотеки для обработки таблиц. Сравним их на одной задаче: **посчитать, сколько записей приходится на каждый уровень Severity**. Оба инструмента читают один и тот же набор файлов Parquet и исключают отсутствующие значения Severity.

- **Pandas** читает файлы по одному и прибавляет полученные числа к общему результату. Это инкрементальная обработка: весь набор в память загружать не нужно.
- **Polars** использует `scan_parquet` и `collect`: сначала строит план вычисления (**lazy execution**), затем выполняет его в потоковом режиме.

Для каждого инструмента сделаем два запуска в разном порядке. Отдельный процесс позволяет измерять память каждого запуска отдельно. Сравним результаты и убедимся, что подсчёты совпадают.

Время включает запуск Python и импорт библиотеки. Пиковая RAM — приблизительный максимум оперативной памяти процесса, который `psutil` считывает каждые 0,02 секунды; краткий пик можно пропустить. Кэш файлов операционной системы, фоновая нагрузка и настройки Colab влияют на измерения. Два запуска дают первое представление, но не являются строгим исследованием производительности.

In [ ]:
"""Сравнение Pandas и Polars на подсчёте ДТП по Severity."""
import json
import subprocess
import sys
import time
import tempfile
from pathlib import Path

import pandas as pd
import psutil


# Этот код запускается отдельно для каждого измерения.
# Так использование памяти одной библиотеки не смешивается с другой.
WORKER_CODE = r'''
import json
import sys
from pathlib import Path

engine, folder = sys.argv[1], sys.argv[2]
files = sorted(str(p) for p in Path(folder).glob('year=*/part-*.parquet'))
if not files:
    raise ValueError('Файлы Parquet не найдены')

if engine == 'Pandas':
    import pandas as pd
    counts = {}
    for file in files:
        frame = pd.read_parquet(file, columns=['Severity'])
        for level, count in frame['Severity'].dropna().value_counts().items():
            counts[str(level)] = counts.get(str(level), 0) + int(count)
elif engine == 'Polars':
    import polars as pl
    # Lazy: Polars сначала строит план вычислений, а выполняет его в collect.
    result = (pl.scan_parquet(files, hive_partitioning=False)
                .select('Severity')
                .filter(pl.col('Severity').is_not_null())
                .group_by('Severity').len().collect(engine='streaming'))
    counts = {str(level): int(count) for level, count in result.iter_rows()}
else:
    raise ValueError('Неизвестная библиотека')
print(json.dumps(counts))
'''


def measure_tool(engine, folder):
    """Запускает отдельный процесс и измеряет время и приблизительный максимум RAM."""
    started = time.perf_counter()
    with tempfile.TemporaryFile(mode='w+', encoding='utf-8') as output, tempfile.TemporaryFile(mode='w+', encoding='utf-8') as errors:
        process = subprocess.Popen(
            [sys.executable, '-c', WORKER_CODE, engine, str(Path(folder).resolve())],
            stdout=output, stderr=errors, text=True,
        )
        tracked = psutil.Process(process.pid)
        peak_bytes = 0
        while process.poll() is None:
            try:
                peak_bytes = max(peak_bytes, tracked.memory_info().rss)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
            time.sleep(0.02)
        output.seek(0)
        errors.seek(0)
        stdout, stderr = output.read(), errors.read()
    seconds = time.perf_counter() - started
    if process.returncode != 0:
        raise RuntimeError(f'{engine}: {stderr}')
    return {'tool': engine, 'seconds': seconds, 'peak_memory_mb': peak_bytes / 1024**2,
            'counts': json.loads(stdout)}


def compare_tools(folder):
    """Проводит два измерения для каждого инструмента в разном порядке."""
    measurements = []
    for run, order in enumerate([['Pandas', 'Polars'], ['Polars', 'Pandas']], start=1):
        for engine in order:
            print(f'Запуск {run}: {engine}', flush=True)
            result = measure_tool(engine, folder)
            result['run'] = run
            measurements.append(result)
    first_counts = measurements[0]['counts']
    if not all(item['counts'] == first_counts for item in measurements):
        raise ValueError('Подсчёты не совпадают! Проверьте данные и запросы.')
    table = pd.DataFrame([{key: value for key, value in item.items() if key != 'counts'}
                          for item in measurements])
    return table, first_counts


In [ ]:
# Выполняем сравнение для уже подготовленных файлов Parquet.
benchmark_results, benchmark_counts = compare_tools(OUTPUT_DIR)

# Для дополнительной проверки сравним результат с первым аудитом.
expected_counts = {str(key): int(value) for key, value in audit["severity"].items() if key != "<missing>"}
assert benchmark_counts == expected_counts, "Результат не совпал с первоначальным подсчётом"
print("Подсчёты Pandas, Polars и первоначального аудита совпадают:", benchmark_counts)

display(benchmark_results.round(2))
benchmark_results.to_csv("/content/traffic_project/pandas_polars_benchmark.csv", index=False)

### Как читать результат

В таблице `seconds` — время запуска в секундах; `peak_memory_mb` — приблизительная максимальная оперативная память процесса в МиБ. Сравниваем оба запуска и учитываем возможный прогрев файлового кэша. Результат относится только к этой конкретной задаче, данным и среде — нельзя утверждать, что один инструмент всегда быстрее другого.

Следующий этап — подготовка выборки для машинного обучения, проверка отсутствия утечки и сравнение baseline с двумя моделями.

## 13. Подготовка данных для машинного обучения

Теперь попробуем предсказать `Severity` — уровень влияния ДТП на движение от 1 до 4. Это **не показатель тяжести травм**. Задача модели: оценить уровень после регистрации ДТП, используя выбранные доступные признаки. Мы не прогнозируем, произойдёт ли ДТП.

Полный набор уже обработан ранее. Для обучения используем около **3% строк**, выбранных воспроизводимо по ID. Это уменьшает время работы и потребление памяти. Каждая строка попадает в выборку независимо от порядка файлов; точный размер напечатаем после отбора. Это не гарантирует одинакового числа примеров каждого редкого класса.

Делим данные по времени:

- **train** — события до 2021 года, на них модель обучается;
- **validation** — события 2021 года, по ним сравниваем модели;
- **test** — 2022–2023 годы, оставляем для финальной оценки после выбора модели.

Границы выбраны заранее. В отличие от случайного перемешивания, такой способ проверяет, насколько модель переносится на более поздний период. Мы также проверяем, что один и тот же ID не попал в разные части. Дубли событий с разными ID эта проверка не обнаружит.

Признаки: штат, час, день недели, день/ночь, температура, влажность, видимость и погодная группа. Конечное время, описание ДТП и другие поля, которые могут содержать сведения о последствиях, в модель не подаём. Погоду используем, только если время измерения не позже начала ДТП и не старше трёх часов. В остальных случаях оставляем пропуск. Это правило снижает риск утечки, но не гарантирует, что каждый признак был реально доступен в системе в момент прогноза.

In [ ]:
"""Подготовка выборки и сравнение моделей без утечки по времени."""
from pathlib import Path
import threading
import time

import numpy as np
import pandas as pd
import psutil
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


BASE_COLUMNS = ['State', 'hour', 'day_of_week', 'Sunrise_Sunset']
WEATHER_NUMBERS = ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)']
WEATHER_CATEGORY = ['weather_group']
FEATURES = BASE_COLUMNS + WEATHER_NUMBERS + WEATHER_CATEGORY
LABELS = [1, 2, 3, 4]


def create_model_sample(folder, sample_percent=3, validation_percent=None, train_percent=None, extra_columns=None):
    """Берём воспроизводимую долю по хешу ID и готовим признаки."""
    files = sorted(Path(folder).glob('year=*/part-*.parquet'))
    if not files:
        raise FileNotFoundError('Файлы Parquet не найдены')
    if not 0 < sample_percent <= 100:
        raise ValueError('sample_percent должен быть от 0 до 100')
    if validation_percent is not None and not 0 < validation_percent <= 100:
        raise ValueError('validation_percent должен быть от 0 до 100')
    if train_percent is not None and not 0 < train_percent <= 100:
        raise ValueError('train_percent должен быть от 0 до 100')
    columns = ['ID', 'Severity', 'Start_Time', 'State', 'Sunrise_Sunset',
               'Weather_Timestamp', 'Weather_Condition'] + WEATHER_NUMBERS
    if extra_columns:
        columns += [name for name in extra_columns if name not in columns]
    frames = []
    for file in files:
        data = pd.read_parquet(file, columns=columns)
        # ID нужен только для отбора и проверки пересечений, в модель он не попадёт.
        valid_id = data['ID'].notna()
        hashes = pd.util.hash_pandas_object(data['ID'], index=False)
        selected = hashes % 10000 < sample_percent * 100
        if train_percent is not None:
            years = pd.to_numeric(data['Start_Time'].str.slice(0, 4), errors='coerce')
            is_train = years < 2021
            selected = selected.where(~is_train, hashes % 10000 < train_percent * 100)
        if validation_percent is not None:
            is_validation = data['Start_Time'].str.startswith('2021-', na=False)
            selected = selected.where(~is_validation, hashes % 10000 < validation_percent * 100)
        data = data.loc[valid_id & selected].copy()
        frames.append(data)
    data = pd.concat(frames, ignore_index=True)
    data['Severity'] = pd.to_numeric(data['Severity'], errors='coerce')
    data['start'] = pd.to_datetime(data['Start_Time'], format='mixed', errors='coerce')
    data['weather_time'] = pd.to_datetime(data['Weather_Timestamp'], format='mixed', errors='coerce')
    data = data.loc[data['Severity'].isin(LABELS) & data['start'].notna()].copy()
    data['Severity'] = data['Severity'].astype(int)
    data['hour'] = data['start'].dt.hour
    data['day_of_week'] = data['start'].dt.dayofweek
    # Погода допустима, только если измерена не позже ДТП и не старше 3 часов.
    age = data['start'] - data['weather_time']
    timely_weather = age.between(pd.Timedelta(0), pd.Timedelta(hours=3))
    data['weather_available'] = timely_weather
    for col in WEATHER_NUMBERS:
        data[col] = pd.to_numeric(data[col], errors='coerce').where(timely_weather)
    # Группировка погоды задана в предыдущем разделе ноутбука.
    data['weather_group'] = data['Weather_Condition'].map(weather_group).where(timely_weather, 'Unknown')
    data['Sunrise_Sunset'] = data['Sunrise_Sunset'].fillna('Unknown')
    data['State'] = data['State'].fillna('Unknown')
    return data


def split_by_time(data):
    """Train: до 2021, validation: 2021, test: 2022–2023."""
    train = data[data['start'].dt.year < 2021].copy()
    validation = data[data['start'].dt.year == 2021].copy()
    test = data[data['start'].dt.year >= 2022].copy()
    if min(len(train), len(validation), len(test)) == 0:
        raise ValueError('Нужны данные всех трёх периодов. Используйте полный набор или другую выборку.')
    assert set(train['ID']).isdisjoint(validation['ID'])
    assert set(train['ID']).isdisjoint(test['ID'])
    assert set(validation['ID']).isdisjoint(test['ID'])
    return train, validation, test


def build_pipeline(model):
    """Заполнение пропусков и кодирование обучаются только на train."""
    numeric = ['hour', 'day_of_week'] + WEATHER_NUMBERS
    categorical = ['State', 'Sunrise_Sunset'] + WEATHER_CATEGORY
    numeric_steps = Pipeline([('fill', SimpleImputer(strategy='median', keep_empty_features=True)),
                              ('scale', StandardScaler())])
    categories = OneHotEncoder(handle_unknown='ignore')
    preparation = ColumnTransformer([('numbers', numeric_steps, numeric),
                                     ('categories', categories, categorical)])
    return Pipeline([('prepare', preparation), ('model', model)])


def fit_with_measurement(model, x, y):
    """Измеряем длительность и примерный пик RAM всего текущего процесса."""
    process = psutil.Process()
    peak = [process.memory_info().rss]
    stop = threading.Event()
    def monitor():
        while not stop.wait(0.02):
            peak[0] = max(peak[0], process.memory_info().rss)
    thread = threading.Thread(target=monitor, daemon=True)
    started = time.perf_counter()
    thread.start()
    try:
        model.fit(x, y)
    finally:
        stop.set()
        thread.join()
    return time.perf_counter() - started, peak[0] / 1024**2


def compare_models(train, validation):
    """Сравниваем только на validation; test остаётся для финальной оценки."""
    models = {
        'Dummy': DummyClassifier(strategy='most_frequent'),
        'Logistic Regression': build_pipeline(LogisticRegression(max_iter=500, random_state=42)),
        'Random Forest': build_pipeline(RandomForestClassifier(n_estimators=50, max_depth=14,
                                      min_samples_leaf=5, n_jobs=2, random_state=42)),
    }
    rows, predictions = [], {}
    for name, model in models.items():
        print('Обучаем:', name, flush=True)
        seconds, memory = fit_with_measurement(model, train[FEATURES], train['Severity'])
        predicted = model.predict(validation[FEATURES])
        predictions[name] = predicted
        rows.append({'model': name, 'accuracy': accuracy_score(validation['Severity'], predicted),
                     'f1_macro': f1_score(validation['Severity'], predicted, labels=LABELS, average='macro', zero_division=0),
                     'train_seconds': seconds, 'peak_process_ram_mib': memory})
    return pd.DataFrame(rows), models, predictions


def improve_models(train, validation):
    """Сравнение моделей с разным учётом редких классов на validation."""
    candidates = {
        'Logistic Regression': build_pipeline(LogisticRegression(max_iter=500, random_state=42)),
        'Logistic Regression balanced': build_pipeline(LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)),
        'Logistic Regression moderate': build_pipeline(LogisticRegression(max_iter=500, class_weight={1: 3, 2: 1, 3: 1.5, 4: 2}, random_state=42)),
        'Logistic Regression weight 3/4 = 2/4': build_pipeline(LogisticRegression(max_iter=500, class_weight={1: 3, 2: 1, 3: 2, 4: 4}, random_state=42)),
        'Logistic Regression weight 3/4 = 2/8': build_pipeline(LogisticRegression(max_iter=500, class_weight={1: 3, 2: 1, 3: 2, 4: 8}, random_state=42)),
        'Logistic Regression weight 3/4 = 3/8': build_pipeline(LogisticRegression(max_iter=500, class_weight={1: 3, 2: 1, 3: 3, 4: 8}, random_state=42)),
        'Random Forest balanced': build_pipeline(RandomForestClassifier(n_estimators=50, max_depth=14,
                  min_samples_leaf=5, class_weight='balanced', n_jobs=2, random_state=42)),
    }
    scores, fitted, predictions = [], {}, {}
    for name, model in candidates.items():
        print('Проверяем:', name, flush=True)
        seconds, memory = fit_with_measurement(model, train[FEATURES], train['Severity'])
        predicted = model.predict(validation[FEATURES])
        scores.append({'model': name,
                       'accuracy': accuracy_score(validation['Severity'], predicted),
                       'f1_macro': f1_score(validation['Severity'], predicted, labels=LABELS, average='macro', zero_division=0),
                       'train_seconds': seconds, 'peak_process_ram_mib': memory})
        fitted[name] = model
        predictions[name] = predicted
    return pd.DataFrame(scores), fitted, predictions


In [ ]:
# Считываем только нужные признаки и берём небольшую долю записей.
# При повторном запуске этой ячейки выборка строится заново, но CSV не обрабатывается заново.
sample = create_model_sample(OUTPUT_DIR, sample_percent=3)
train, validation, test = split_by_time(sample)
print("Размер выборки:", len(sample))
print("Train:", len(train), "| validation:", len(validation), "| test:", len(test))
print("Доля строк с погодным измерением, подходящим по времени:", round(sample["weather_available"].mean() * 100, 1), "%")

# Посмотрим, сколько примеров каждого класса находится в каждом периоде.
split_counts = pd.DataFrame({
    "train": train["Severity"].value_counts(),
    "validation": validation["Severity"].value_counts(),
    "test": test["Severity"].value_counts()
}).fillna(0).astype(int).reindex(LABELS, fill_value=0)
display(split_counts)

## 14. Сравнение трёх моделей

1. **DummyClassifier** — baseline: всегда выбирает наиболее частый класс из train.
2. **Logistic Regression** — линейная модель классификации.
3. **Random Forest** — модель из набора деревьев решений; ограничиваем её размер для скорости.

Внутри `Pipeline` пропуски числовых признаков заполняются медианой, числовые значения масштабируются, а текстовые категории превращаются в числовые столбцы (**one-hot encoding**). Эти преобразования обучаются только на train и применяются к validation; так информация из validation не попадает в обучение. Для новых категорий в validation используется `handle_unknown="ignore"`.

Сравниваем **Accuracy** — долю всех правильных ответов — и **F1-macro** — среднее F1 по четырём классам с одинаковым весом. F1-macro особенно полезна здесь, потому что классы представлены неравномерно. Также измеряем время и примерный пик RAM. RAM измеряется у всего текущего процесса Python, поэтому включает уже загруженные данные и результаты предыдущих шагов. Эти значения приблизительные и не означают память одной только модели; порядок запуска тоже может влиять.

In [ ]:
model_results, trained_models, validation_predictions = compare_models(train, validation)
display(model_results.round(4))
model_results.to_csv("/content/traffic_project/model_validation.csv", index=False)

# Выбираем модель по F1-macro на validation, а не на test.
best_name = model_results.sort_values("f1_macro", ascending=False).iloc[0]["model"]
print("Лучшая модель на validation по F1-macro:", best_name)

## 15. Ошибки на validation

**Confusion matrix** — таблица ошибок: строки показывают настоящий класс, столбцы — предсказанный. Значения на диагонали означают правильные ответы. В отчёте ниже `support` показывает число реальных примеров каждого класса. Если модель почти всегда выбирает один класс, Accuracy может быть высокой, хотя качество на редких классах будет низким.

Пока анализируем только validation. После выбора подхода оценим test отдельно и один раз, не используя его для настройки модели.

In [ ]:
prediction = validation_predictions[best_name]
print(classification_report(validation["Severity"], prediction, labels=LABELS, zero_division=0))
matrix = confusion_matrix(validation["Severity"], prediction, labels=LABELS)
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matrix, cmap="Blues")
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, str(matrix[i, j]), ha="center", va="center", color="black")
ax.set_xticks(range(4), LABELS)
ax.set_yticks(range(4), LABELS)
ax.set_xlabel("Предсказанный Severity")
ax.set_ylabel("Настоящий Severity")
ax.set_title("Confusion matrix: validation")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

Далее: сравнить модель с погодными признаками и без них на одинаковом разбиении, разобрать ошибки по погодным группам и провести финальную оценку test. В конце проекта отдельно проверим полный список требований преподавателя.

## 16. Почему модели ошибаются на редких классах

Посмотрим отдельно на каждый класс, а не только на общую Accuracy. **Recall** показывает, какую долю настоящих ДТП определённого класса модель распознала. **Precision** показывает, какая доля предсказаний этого класса оказалась правильной. F1 объединяет precision и recall.

Классы могут встречаться очень неравномерно. Например, модель, которая почти всегда отвечает «2», получает много правильных ответов, но пропускает редкие классы. Посмотрим на отчёты первой логистической регрессии и baseline и на число предсказаний каждого класса.

In [ ]:
for name in ["Dummy", "Logistic Regression"]:
    print("\nМодель:", name)
    pred = validation_predictions[name]
    print(classification_report(validation["Severity"], pred, labels=LABELS, zero_division=0))
    print("Число предсказаний по классам:")
    print(pd.Series(pred).value_counts().reindex(LABELS, fill_value=0).sort_index())

## 17. Проверим веса классов

У модели можно увеличить `class_weight`: ошибка на более редком классе получает больший вес при обучении. Мы проверим автоматический вариант `balanced` и несколько простых наборов весов, выбранных вручную. Сравним их с обычной Logistic Regression и Random Forest.

Важно: увеличение веса не гарантирует улучшения. Модель может начать находить больше редких случаев, но при этом чаще ошибаться на других. Поэтому смотрим на F1-macro, Accuracy и метрики каждого класса. Измерения выполняются на той же validation-выборке, что и раньше.

In [ ]:
improvement_results, improved_models, improved_predictions = improve_models(train, validation)
# Добавляем исходные Dummy и Random Forest, чтобы они тоже участвовали в сравнении.
previous = model_results[model_results["model"].isin(["Dummy", "Random Forest"])]
all_results = pd.concat([previous, improvement_results], ignore_index=True)
all_results = all_results.sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(all_results.round(4))
all_results.to_csv("/content/traffic_project/model_improvements.csv", index=False)

# Сохраним выбранную модель в переменной — она понадобится на следующем этапе.
chosen_name = all_results.iloc[0]["model"]
if chosen_name in improved_models:
    chosen_model = improved_models[chosen_name]
    chosen_predictions = improved_predictions[chosen_name]
else:
    chosen_model = trained_models[chosen_name]
    chosen_predictions = validation_predictions[chosen_name]
print("Выбор по F1-macro на validation:", chosen_name)

### Проверка ошибок после изменения весов

Сравним метрики каждого класса и матрицу ошибок. Обратите внимание на `support`: если в validation очень мало примеров какого-то класса, его метрики могут сильно измениться даже из-за одной записи. Например, при двух примерах доля правильных ответов может изменяться скачками по 50 процентных пунктов.

Мы попробовали несколько вариантов на одной validation-выборке, поэтому выбранный F1-macro может оказаться завышенным для новых данных. Test пока не использовался при настройке. На следующем этапе проверим выбранный подход на test и честно покажем результат.

In [ ]:
print("Выбранная модель:", chosen_name)
print(classification_report(validation["Severity"], chosen_predictions, labels=LABELS, zero_division=0))

# Нормируем каждую строку: теперь видно процент предсказаний внутри настоящего класса.
matrix = confusion_matrix(validation["Severity"], chosen_predictions, labels=LABELS, normalize="true")
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        color = "white" if matrix[i, j] > 0.5 else "black"
        ax.text(j, i, f"{matrix[i, j]:.1%}", ha="center", va="center", color=color)
ax.set_xticks(range(4), LABELS)
ax.set_yticks(range(4), LABELS)
ax.set_xlabel("Предсказанный Severity")
ax.set_ylabel("Настоящий Severity")
ax.set_title("Ошибки по классам: validation")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

## 18. Проверка идей из примеров Kaggle

В нескольких изученных ноутбуках используются координаты места ДТП, месяц, наличие перекрёстка и светофора. Проверим, помогают ли эти признаки **наших данных**. Признаки, отражающие состояние инфраструктуры, считаем известными после получения координат; на практике это предположение нужно проверять для конкретной системы.

Изученные примеры:

- `ml-to-predict-accident-severity-pa-mont.ipynb` — четыре модели на данных округа Montgomery в Pennsylvania; используемые там длительность ДТП и расстояние потенциально недоступны при раннем прогнозе, поэтому мы их не добавляем.
- `usa-car-accidents-severity-prediction.ipynb` — признаки дорожной среды и уменьшение большинства в обучении. Там решается другая задача: Severity 4 против остальных.
- `60-insights-extraction-us-accident-analysis.ipynb` — идеи для графиков по времени, месту и погоде.

Здесь проверяем координаты начала, месяц, переход, перекрёсток, знак Stop и светофор. Прогноз делаем с той же целью и на том же временном разбиении. Признаки не гарантируют улучшение: проверим его по validation.

In [ ]:
"""Проверка идей из примеров Kaggle на прежнем временном разбиении."""
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


EXTRA_NUMERIC = ['Start_Lat', 'Start_Lng', 'month']
EXTRA_CATEGORIES = ['Crossing', 'Junction', 'Traffic_Signal', 'Stop']
EXTENDED_FEATURES = FEATURES + EXTRA_NUMERIC + EXTRA_CATEGORIES


def add_kaggle_features(sample, folder):
    """Добавляем данные только по ID уже выбранных записей."""
    ids = set(sample['ID'])
    columns = ['ID', 'Start_Lat', 'Start_Lng'] + EXTRA_CATEGORIES
    pieces = []
    for file in sorted(Path(folder).glob('year=*/part-*.parquet')):
        data = pd.read_parquet(file, columns=columns)
        pieces.append(data[data['ID'].isin(ids)])
    extra = pd.concat(pieces, ignore_index=True)
    if extra['ID'].duplicated().any():
        raise ValueError('Повтор ID — невозможно однозначно соединить данные.')
    data = sample.merge(extra, on='ID', how='left', validate='one_to_one')
    if len(data) != len(sample):
        raise ValueError('При соединении изменилось число строк')
    data['month'] = data['start'].dt.month
    for name in ['Start_Lat', 'Start_Lng']:
        data[name] = pd.to_numeric(data[name], errors='coerce')
    for name in EXTRA_CATEGORIES:
        data[name] = data[name].fillna('Unknown').astype(str)
    return data


def make_extended_pipeline(model):
    """Заполнение пропусков и подготовка новых признаков только на train."""
    numbers = ['hour', 'day_of_week'] + WEATHER_NUMBERS + EXTRA_NUMERIC
    categories = ['State', 'Sunrise_Sunset'] + WEATHER_CATEGORY + EXTRA_CATEGORIES
    numeric_steps = Pipeline([('fill', SimpleImputer(strategy='median', keep_empty_features=True)),
                              ('scale', StandardScaler())])
    prepare = ColumnTransformer([('numbers', numeric_steps, numbers),
                                 ('categories', OneHotEncoder(handle_unknown='ignore'), categories)])
    return Pipeline([('prepare', prepare), ('model', model)])


def test_kaggle_ideas(train, validation):
    """Сравниваем дополнительные признаки и модели только на validation."""
    options = {
        'Logistic + location / road': LogisticRegression(max_iter=500, random_state=42,
                                      class_weight={1: 3, 2: 1, 3: 2, 4: 8}),
        'Random Forest + location / road': RandomForestClassifier(n_estimators=100, max_depth=18,
                                             min_samples_leaf=5, n_jobs=2, random_state=42,
                                             class_weight={1: 3, 2: 1, 3: 2, 4: 8}),
    }
    rows, models, predictions = [], {}, {}
    for name, estimator in options.items():
        print('Обучаем:', name, flush=True)
        model = make_extended_pipeline(estimator)
        seconds, memory = fit_with_measurement(model, train[EXTENDED_FEATURES], train['Severity'])
        pred = model.predict(validation[EXTENDED_FEATURES])
        rows.append({'model': name, 'accuracy': accuracy_score(validation['Severity'], pred),
                     'f1_macro': f1_score(validation['Severity'], pred, labels=LABELS, average='macro', zero_division=0),
                     'train_seconds': seconds, 'peak_process_ram_mib': memory})
        models[name], predictions[name] = model, pred
    return pd.DataFrame(rows), models, predictions


In [ ]:
# Дополняем уже существующую выборку по ID — строки остаются теми же.
extended_sample = add_kaggle_features(sample, OUTPUT_DIR)
extended_train, extended_validation, extended_test = split_by_time(extended_sample)
extra_results, extra_models, extra_predictions = test_kaggle_ideas(extended_train, extended_validation)

# Сравниваем с лучшим из предыдущих вариантов на тех же 3% validation.
print("Предыдущие модели:")
display(all_results[["model", "accuracy", "f1_macro"]].head())
print("Модели с дополнительными признаками:")
display(extra_results.round(4))
extra_results.to_csv("/content/traffic_project/kaggle_feature_validation.csv", index=False)

### Более крупная validation-выборка

В выборке 3% в 2021 году примеров Severity 1 может быть очень мало. Для дополнительной проверки возьмём **20% всех классов 2021 года** с помощью того же правила отбора по ID. Train и test останутся на уровне 3%, как раньше. Увеличение одинаково применяется ко всем классам validation, чтобы сохранить их естественные пропорции насколько позволяет случайная выборка.

Сравниваем прежнюю модель и модели с дополнительными признаками на одной расширенной validation. Мы уже рассматривали разные варианты на 2021 году, поэтому это проверка устойчивости, а не полностью независимая оценка. Test за 2022–2023 годы по-прежнему не используется для выбора.

In [ ]:
# Эта ячейка повторно читает подготовленные Parquet и может потребовать больше RAM.
large_sample = create_model_sample(OUTPUT_DIR, sample_percent=3, validation_percent=20)
large_train, large_validation, _ = split_by_time(large_sample)
print("Train:", len(large_train), "| validation:", len(large_validation))
print("Классы validation:")
print(large_validation["Severity"].value_counts().reindex(LABELS, fill_value=0).sort_index())

# Сравниваем ту же модель, что дала лучший результат в нашем предыдущем запуске.
old_model = build_pipeline(LogisticRegression(max_iter=500, random_state=42,
                                               class_weight={1: 3, 2: 1, 3: 2, 4: 8}))
old_model.fit(large_train[FEATURES], large_train["Severity"])
old_pred = old_model.predict(large_validation[FEATURES])
print("Без новых признаков: Accuracy", round(accuracy_score(large_validation["Severity"], old_pred), 4),
      "F1-macro", round(f1_score(large_validation["Severity"], old_pred, labels=LABELS, average="macro", zero_division=0), 4))
print(classification_report(large_validation["Severity"], old_pred, labels=LABELS, zero_division=0))

large_extended = add_kaggle_features(large_sample, OUTPUT_DIR)
large_train_extra, large_val_extra, _ = split_by_time(large_extended)
large_extra_results, _, _ = test_kaggle_ideas(large_train_extra, large_val_extra)
display(large_extra_results.round(4))

**Как сделать вывод:** сравните значения F1-macro на одной и той же выборке. Если дополнительные признаки не помогли, нет необходимости добавлять их в итоговую модель. Отдельно посмотрите, сколько реальных примеров Severity 1 распознано: высокий общий показатель не решает проблему редкого класса. Финальную оценку на test выполним после выбора окончательного подхода.

## 19. Дополнительная проверка дисбаланса и Gradient Boosting

В следующих примерах Kaggle мы изучили нейросети и Gradient Boosting. Примеры нейросетей в основном показывают Accuracy; один из них использует признаки завершения ДТП и случайное разделение данных. Поэтому напрямую сравнивать их Accuracy с нашей F1-macro нельзя.

Проверим две идеи на **той же расширенной validation-выборке** 2021 года:

1. Оставим только часть записей самого частого класса 2 в train. Другие классы оставим целиком. Этот подход называется **undersampling**; validation не меняется.
2. Обучим `HistGradientBoostingClassifier` — модель, которая последовательно строит небольшие деревья решений. Проверим вариант без весов и с весами классов.

Для честного сравнения используем те же исходные признаки и границы периодов. Редкие классы в validation остаются редкими; в таблице показываем Accuracy и F1-macro.

In [ ]:
"""Последние простые эксперименты с дисбалансом и градиентным бустингом."""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def undersample_majority(train, fraction, seed=42):
    """Уменьшаем класс 2 только в train. Остальные классы не меняем."""
    if not 0 < fraction <= 1:
        raise ValueError('Доля должна быть от 0 до 1')
    common = train[train['Severity'] == 2].sample(frac=fraction, random_state=seed)
    rare = train[train['Severity'] != 2]
    return pd.concat([common, rare]).sample(frac=1, random_state=seed)


def make_boosting_pipeline():
    """Небольшой градиентный бустинг на тех же признаках, что и ранее."""
    numeric = ['hour', 'day_of_week'] + WEATHER_NUMBERS
    categorical = ['State', 'Sunrise_Sunset'] + WEATHER_CATEGORY
    preparation = ColumnTransformer([
        ('numbers', SimpleImputer(strategy='median', keep_empty_features=True), numeric),
        ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
    ], sparse_threshold=0)
    return Pipeline([('prepare', preparation),
                     ('model', HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=15,
                               learning_rate=0.08, random_state=42))])


def test_balancing_and_boosting(train, validation):
    """Сравниваем варианты на неизменённой validation, test не используем."""
    variants = [
        ('Logistic: 50% класса 2', build_pipeline(LogisticRegression(max_iter=500, random_state=42)), 0.5, False),
        ('Logistic: 25% класса 2', build_pipeline(LogisticRegression(max_iter=500, random_state=42)), 0.25, False),
        ('Forest: 25% класса 2', build_pipeline(RandomForestClassifier(n_estimators=50, max_depth=14,
                                min_samples_leaf=5, n_jobs=2, random_state=42)), 0.25, False),
        ('Boosting: обычный', make_boosting_pipeline(), 1.0, False),
        ('Boosting: веса классов', make_boosting_pipeline(), 1.0, True),
    ]
    results, fitted, predictions = [], {}, {}
    for name, model, fraction, weighted in variants:
        print('Обучаем:', name, flush=True)
        subset = undersample_majority(train, fraction)
        if weighted:
            # Веса применяются только во время обучения, без изменения validation.
            weights = subset['Severity'].map({1: 3, 2: 1, 3: 2, 4: 8}).to_numpy()
            import time
            import psutil
            start = time.perf_counter()
            model.fit(subset[FEATURES], subset['Severity'], model__sample_weight=weights)
            seconds = time.perf_counter() - start
            memory = np.nan  # Отдельное измерение пика здесь не проводим.
        else:
            seconds, memory = fit_with_measurement(model, subset[FEATURES], subset['Severity'])
        predicted = model.predict(validation[FEATURES])
        results.append({'model': name, 'train_rows': len(subset),
                        'accuracy': accuracy_score(validation['Severity'], predicted),
                        'f1_macro': f1_score(validation['Severity'], predicted,
                                             labels=LABELS, average='macro', zero_division=0),
                        'train_seconds': seconds, 'peak_process_ram_mib': memory})
        fitted[name], predictions[name] = model, predicted
    return pd.DataFrame(results), fitted, predictions


In [ ]:
# large_train и large_validation подготовлены в предыдущем разделе.
last_results, last_models, last_predictions = test_balancing_and_boosting(large_train, large_validation)
print("Сравнение новых вариантов:")
display(last_results.round(4))
last_results.to_csv("/content/traffic_project/last_experiments.csv", index=False)

# old_model использует предыдущие веса и те же train и validation.
old_pred = old_model.predict(large_validation[FEATURES])
print("Предыдущая модель — Accuracy:", round(accuracy_score(large_validation["Severity"], old_pred), 4),
      "F1-macro:", round(f1_score(large_validation["Severity"], old_pred,
                                   labels=LABELS, average="macro", zero_division=0), 4))

### Метрики каждого класса

Отчёт ниже относится к лучшему из новых вариантов. При сравнении с предыдущей моделью используйте тот же validation. Значение `support` — число настоящих примеров класса. Даже если общая метрика выросла, необходимо проверить, распознаёт ли модель Severity 1 и 4.

RAM показана приблизительно для всего процесса Python, поэтому она включает данные и предыдущие вычисления; для бустинга с весами пик отдельно не измерялся.

In [ ]:
last_best = last_results.sort_values("f1_macro", ascending=False).iloc[0]["model"]
print("Лучший новый вариант:", last_best)
print(classification_report(large_validation["Severity"], last_predictions[last_best],
                            labels=LABELS, zero_division=0))

Мы пробовали несколько настроек на 2021 году, поэтому эта validation уже использована для выбора. Финальный test нужен для независимой оценки. Далее зафиксируем подход, проверим влияние погодных признаков и проанализируем ошибки по погоде.

## 20. Ещё одна модель: Extra Trees

**Extra Trees** — модель из множества деревьев решений. Она похожа на Random Forest, но при построении деревьев использует больше случайности. Это может помочь находить сложные сочетания признаков.

Проверим два варианта: обычный и с весами редких классов. Train и validation, признаки и метрики те же, что в предыдущем разделе. Поэтому результаты можно сравнить с Logistic Regression. Test остаётся для финальной проверки.

In [ ]:
"""Сравнение Extra Trees с предыдущей моделью на validation."""
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score


def try_extra_trees(train, validation):
    """Обучает два варианта Extra Trees на одних и тех же признаках."""
    weights = {1: 3, 2: 1, 3: 2, 4: 8}
    variants = {
        'Extra Trees': ExtraTreesClassifier(n_estimators=100, max_depth=18,
                          min_samples_leaf=5, n_jobs=2, random_state=42),
        'Extra Trees с весами': ExtraTreesClassifier(n_estimators=100, max_depth=18,
                          min_samples_leaf=5, class_weight=weights, n_jobs=2, random_state=42),
    }
    rows, fitted, predictions = [], {}, {}
    for name, estimator in variants.items():
        print('Обучаем:', name, flush=True)
        model = build_pipeline(estimator)
        seconds, memory = fit_with_measurement(model, train[FEATURES], train['Severity'])
        predicted = model.predict(validation[FEATURES])
        rows.append({'model': name, 'accuracy': accuracy_score(validation['Severity'], predicted),
                     'f1_macro': f1_score(validation['Severity'], predicted, labels=LABELS,
                                          average='macro', zero_division=0),
                     'train_seconds': seconds, 'peak_process_ram_mib': memory})
        fitted[name], predictions[name] = model, predicted
    return pd.DataFrame(rows), fitted, predictions


In [ ]:
extra_trees_results, extra_trees_models, extra_trees_predictions = try_extra_trees(large_train, large_validation)
display(extra_trees_results.round(4))
extra_trees_results.to_csv("/content/traffic_project/extra_trees_validation.csv", index=False)

# Сравниваем с предыдущей моделью на том же validation.
print("Logistic Regression — F1-macro:", round(f1_score(large_validation["Severity"], old_pred,
      labels=LABELS, average="macro", zero_division=0), 4))
for name, predicted in extra_trees_predictions.items():
    print("\n", name)
    print(classification_report(large_validation["Severity"], predicted, labels=LABELS, zero_division=0))

Выбор модели делаем по validation и учитываем качество каждого класса, а не только общее число. Поскольку мы пробовали несколько моделей на одной validation, окончательный результат необходимо отдельно проверить на test. Пик памяти в таблице относится ко всему процессу Python и зависит от выполненных ранее ячеек.

## Интерактивное меню и сохранение агрегатов

Это консольный интерфейс для основных операций: можно выбрать год или штат и посмотреть количество ДТП по уровням Severity. Он использует данные из SQLite. Также сохраним рассчитанные агрегаты в Parquet и SQLite — это требование Cluster 8.

Чтобы открыть меню, замените `RUN_MENU = False` на `RUN_MENU = True` в последней ячейке и запустите её. Для выхода введите `0`. По умолчанию меню отключено, чтобы выполнение остальных ячеек не останавливалось в ожидании ввода.

In [ ]:
"""Простое меню для просмотра данных SQLite с фильтрами."""
import sqlite3
from pathlib import Path
import pandas as pd


def save_aggregates(database_path, results, output_folder):
    """Сохраняем уже рассчитанные агрегаты в SQLite и Parquet."""
    folder = Path(output_folder)
    folder.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(database_path) as connection:
        for name in ['severity', 'state', 'year', 'hour', 'weather']:
            table = results[name]
            table.to_sql('agg_' + name, connection, if_exists='replace', index=False)
            table.to_parquet(folder / (name + '.parquet'), index=False)
    print('Агрегаты сохранены в SQLite и Parquet.')


def run_menu(database_path):
    """Показывает статистику по выбранному году или штату."""
    try:
        with sqlite3.connect(database_path) as connection:
            while True:
                print('\n1 — ДТП по Severity за год')
                print('2 — ДТП по Severity в штате')
                print('0 — выход')
                choice = input('Выберите действие: ').strip()
                if choice == '0':
                    print('Выход из меню.')
                    break
                elif choice == '1':
                    year = input('Введите год, например 2021: ').strip()
                    if not (year.isdigit() and len(year) == 4):
                        print('Введите год из четырёх цифр.')
                        continue
                    query = '''SELECT severity AS Severity, COUNT(*) AS accidents
                               FROM accidents WHERE SUBSTR(start_time, 1, 4) = ?
                               GROUP BY severity ORDER BY severity'''
                    result = pd.read_sql_query(query, connection, params=(year,))
                elif choice == '2':
                    state = input('Введите код штата, например CA: ').strip().upper()
                    query = '''SELECT a.severity AS Severity, COUNT(*) AS accidents
                               FROM accidents a JOIN states s ON a.state_id = s.state_id
                               WHERE s.state_code = ? GROUP BY a.severity ORDER BY a.severity'''
                    result = pd.read_sql_query(query, connection, params=(state,))
                else:
                    print('Неизвестное действие. Выберите 0, 1 или 2.')
                    continue
                if result.empty:
                    print('Данные не найдены.')
                else:
                    print(result.to_string(index=False))
    except (sqlite3.Error, OSError) as error:
        print('Не удалось открыть базу:', error)


In [ ]:
# Сохраняем готовые таблицы. При повторном запуске обновятся только агрегаты.
save_aggregates(DATABASE_PATH, results, "/content/traffic_project/aggregates")

In [ ]:
# Поставьте True и запустите эту ячейку для работы с меню.
RUN_MENU = False
if RUN_MENU:
    run_menu(DATABASE_PATH)
else:
    print("Меню отключено. Чтобы открыть его, установите RUN_MENU = True.")

## 20. Больше обучающих данных и баланс классов

Для обучения используем **30% записей до 2021 года** — в десять раз больше прежней доли. Для сравнения сохраняем ту же validation-выборку: 20% записей за 2021 год с её исходным распределением классов. Test оставляем для финальной проверки.

Проверим два способа работы с дисбалансом:

1. **Одинаковое число строк:** берём по 50 000 записей каждого класса из 30%-ной выборки. Редкие классы повторяются, а часть частых записей не используется. Повторные строки не являются новыми уникальными ДТП.
2. **Веса классов:** используем все выбранные строки. При `class_weight="balanced"` классы получают примерно одинаковый общий вес в обучении. Вариант с умеренными весами оставляет более частым классам больший вес.

Балансируем только train. Если искусственно выровнять validation, оценки не будут отражать распределение исходных данных. Этот этап может потребовать несколько гигабайт оперативной памяти и занять больше времени.

In [ ]:
"""Обучение на более крупной выборке и балансировка только train."""
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score


def balance_train(train, target_per_class=50000, seed=42):
    """Сэмплируем одинаковое число на класс; редкие записи повторяются."""
    groups = []
    for level in LABELS:
        subset = train[train['Severity'] == level]
        if subset.empty:
            raise ValueError(f'В train нет класса {level}')
        groups.append(subset.sample(n=target_per_class, replace=len(subset) < target_per_class, random_state=seed))
    return pd.concat(groups).sample(frac=1, random_state=seed).reset_index(drop=True)


def compare_large_training(train, validation):
    """Одна исходная validation, одинаковый train для сравнения моделей."""
    balanced = balance_train(train)
    print('До балансировки:\n', train['Severity'].value_counts().sort_index(), flush=True)
    print('После балансировки:\n', balanced['Severity'].value_counts().sort_index(), flush=True)
    print('Уникальных ID после балансировки:', balanced['ID'].nunique(), flush=True)
    options = {
        'Logistic 30% balanced': build_pipeline(LogisticRegression(max_iter=500, random_state=42)),
        'Forest 30% balanced': build_pipeline(RandomForestClassifier(n_estimators=100, max_depth=18,
                                     min_samples_leaf=5, n_jobs=2, random_state=42)),
    }
    rows, models, predictions = [], {}, {}
    for name, model in options.items():
        print('Обучаем:', name, flush=True)
        seconds, memory = fit_with_measurement(model, balanced[FEATURES], balanced['Severity'])
        predicted = model.predict(validation[FEATURES])
        rows.append({'model': name, 'accuracy': accuracy_score(validation['Severity'], predicted),
                     'f1_macro': f1_score(validation['Severity'], predicted, labels=LABELS, average='macro', zero_division=0),
                     'train_seconds': seconds, 'peak_process_ram_mib': memory})
        models[name], predictions[name] = model, predicted
    return pd.DataFrame(rows), models, predictions, balanced


def compare_full_training(train, validation):
    """Обучаем на всех строках 30% с разными весами классов."""
    options = {
        'Logistic 30% равные веса классов': 'balanced',
        'Logistic 30% умеренные веса': {1: 3, 2: 1, 3: 2, 4: 8},
    }
    rows, models, predictions = [], {}, {}
    for name, weights in options.items():
        print('Обучаем на всех строках train:', name, flush=True)
        model = build_pipeline(LogisticRegression(max_iter=500, class_weight=weights, random_state=42))
        seconds, memory = fit_with_measurement(model, train[FEATURES], train['Severity'])
        pred = model.predict(validation[FEATURES])
        rows.append({'model': name, 'train_rows': len(train),
                     'accuracy': accuracy_score(validation['Severity'], pred),
                     'f1_macro': f1_score(validation['Severity'], pred, labels=LABELS, average='macro', zero_division=0),
                     'train_seconds': seconds, 'peak_process_ram_mib': memory})
        models[name], predictions[name] = model, pred
    return pd.DataFrame(rows), models, predictions


In [ ]:
# Берём 30% обучующего периода, 20% validation и 3% test.
large30 = create_model_sample(OUTPUT_DIR, sample_percent=3, validation_percent=20, train_percent=30)
train30, val20, test3 = split_by_time(large30)
print("Train:", len(train30), "| validation:", len(val20), "| test:", len(test3))
print("Классы train до балансировки:")
print(train30["Severity"].value_counts().reindex(LABELS, fill_value=0).sort_index())
print("Классы validation:")
print(val20["Severity"].value_counts().reindex(LABELS, fill_value=0).sort_index())

In [ ]:
# Проверка на одинаковом количестве строк каждого класса.
balanced30_results, balanced30_models, balanced30_predictions, balanced30_data = compare_large_training(train30, val20)
display(balanced30_results.round(4))
balanced30_results.to_csv("/content/traffic_project/balanced30_results.csv", index=False)

In [ ]:
# Проверка на всех 30% обучающих строк с весами классов.
full30_results, full30_models, full30_predictions = compare_full_training(train30, val20)
display(full30_results.round(4))
full30_results.to_csv("/content/traffic_project/full30_results.csv", index=False)

# Отчёт для каждого подхода: важно проверить все классы отдельно.
for name, prediction in {**balanced30_predictions, **full30_predictions}.items():
    print("\n", name)
    print(classification_report(val20["Severity"], prediction, labels=LABELS, zero_division=0))

Сравнивайте F1-macro и Accuracy вместе, а также precision и recall каждого класса. При очень редком классе высокие веса или повторение строк могут повысить recall, одновременно увеличив число ложных тревог. Выбор делаем на validation; test здесь не оценивается.

## 21. Более крупные выборки и stratify

Этот отдельный эксперимент использует 30% записей за все годы и случайно делит их на train, validation и test в соотношении 60/20/20. Параметр `stratify` сохраняет примерно одинаковые **доли классов** в трёх частях. Например, если класс занимает 10% исходных данных, он будет занимать около 10% каждой части. Это само по себе не делает классы равными.

Чтобы проверить результат, когда каждый класс одинаково представлен, дополнительно формируем **равные по размеру части validation и test**: из каждого класса без повторений выбирается столько записей, сколько есть в самом редком классе. Сохраняем и полные части с естественным распределением — они показывают, как модель работает при исходной частоте событий. Сначала сравниваем модели только на validation.

Train балансируется отдельно, после разделения: по 50 000 строк каждого класса. Если уникальных редких записей меньше, некоторые повторяются; это не создаёт новых уникальных ДТП. Проверяем также вариант со всеми строками train и весами классов.

**Ограничение:** случайное разделение и временное разделение отвечают на разные вопросы. Случайное показывает качество на похожих по периоду данных, а не гарантирует качество на будущих годах. Ранее мы использовали часть 2021 года для настройки; некоторые из этих записей могут оказаться в новых частях, поэтому новую test-выборку нельзя считать полностью независимой от всех предыдущих экспериментов. Повторы одного ID исключаются, однако похожие записи одного ДТП с разными ID всё ещё возможны.

In [ ]:
"""Эксперимент: стратифицированное разбиение и баланс классов."""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score


def stratified_split(sample):
    """60/20/20 со сходными долями Severity; test оставляем отдельно."""
    if sample['ID'].duplicated().any():
        raise ValueError('Есть повторяющиеся ID — сначала нужно проверить дубликаты.')
    train, rest = train_test_split(sample, test_size=0.4, random_state=42,
                                   stratify=sample['Severity'])
    validation, test = train_test_split(rest, test_size=0.5, random_state=42,
                                        stratify=rest['Severity'])
    assert set(train.ID).isdisjoint(validation.ID)
    assert set(train.ID).isdisjoint(test.ID)
    assert set(validation.ID).isdisjoint(test.ID)
    return train, validation, test


def balanced_evaluation(validation):
    """Равная оценка без повторов: берём столько строк, сколько у редчайшего класса."""
    counts = validation['Severity'].value_counts().reindex(LABELS, fill_value=0)
    count = int(counts.min())
    if count == 0:
        raise ValueError('В validation отсутствует один из классов')
    return (validation.groupby('Severity', group_keys=False)
            .sample(n=count, random_state=42)
            .sample(frac=1, random_state=42))


def evaluate_stratified(train, validation):
    """Подбираем на validation, test не передаётся в эту функцию."""
    balanced_train = balance_train(train, target_per_class=50000)
    balanced_val = balanced_evaluation(validation)
    print('Состав train после балансировки:', balanced_train.Severity.value_counts().sort_index().to_dict(), flush=True)
    print('Уникальные ID в train после балансировки:', balanced_train.ID.nunique(), flush=True)
    print('Состав равной validation:', balanced_val.Severity.value_counts().sort_index().to_dict(), flush=True)
    models = {
        'Logistic — balanced train': build_pipeline(LogisticRegression(max_iter=500, random_state=42)),
        'Random Forest — balanced train': build_pipeline(RandomForestClassifier(n_estimators=100, max_depth=18, min_samples_leaf=5, n_jobs=2, random_state=42)),
        'Logistic — all train, moderate weights': build_pipeline(LogisticRegression(max_iter=500, random_state=42, class_weight={1:3, 2:1, 3:2, 4:8})),
    }
    rows, fitted, predictions = [], {}, {}
    for name, model in models.items():
        subset = train if 'all train' in name else balanced_train
        print('Обучаем:', name, flush=True)
        seconds, memory = fit_with_measurement(model, subset[FEATURES], subset['Severity'])
        natural_prediction = model.predict(validation[FEATURES])
        balanced_prediction = model.predict(balanced_val[FEATURES])
        rows.append({'model': name, 'train_rows': len(subset),
                     'accuracy_natural': accuracy_score(validation.Severity, natural_prediction),
                     'f1_macro_natural': f1_score(validation.Severity, natural_prediction, labels=LABELS, average='macro', zero_division=0),
                     'accuracy_equal_classes': accuracy_score(balanced_val.Severity, balanced_prediction),
                     'f1_macro_equal_classes': f1_score(balanced_val.Severity, balanced_prediction, labels=LABELS, average='macro', zero_division=0),
                     'train_seconds': seconds, 'peak_process_ram_mib': memory})
        fitted[name] = model
        predictions[name] = (natural_prediction, balanced_prediction)
    return pd.DataFrame(rows), fitted, predictions, balanced_val, balanced_train


In [ ]:
# Текущие большие переменные из прошлых экспериментов можно удалить вручную,
# если памяти Colab не хватает. Повторная обработка CSV не нужна.
strat_sample = create_model_sample(OUTPUT_DIR, sample_percent=30)
strat_train, strat_validation, strat_test = stratified_split(strat_sample)
print("Train:", len(strat_train), "| validation:", len(strat_validation), "| test:", len(strat_test))
counts = pd.DataFrame({"train": strat_train.Severity.value_counts(),
                       "validation": strat_validation.Severity.value_counts(),
                       "test": strat_test.Severity.value_counts()}).reindex(LABELS, fill_value=0)
display(counts)

# Равная test-выборка создаётся сейчас, но оценки на ней пока не считаем.
equal_test = balanced_evaluation(strat_test)
print("Test с одинаковым количеством строк по классам:")
print(equal_test.Severity.value_counts().sort_index())

In [ ]:
strat_results, strat_models, strat_predictions, equal_validation, balanced_train = evaluate_stratified(strat_train, strat_validation)
display(strat_results.round(4))
strat_results.to_csv("/content/traffic_project/stratified_validation.csv", index=False)
print("Валидация: равное число по классам", equal_validation.Severity.value_counts().sort_index().to_dict())

В таблице столбцы `natural` показывают оценку на полном validation с исходными долями классов, а `equal_classes` — на validation с одинаковым количеством каждого класса. Эти показатели оценивают разные условия; сравнивать их можно только вместе с описанием состава выборок. `F1-macro` по своей формуле уже даёт каждому классу одинаковый вес, но precision отдельных классов зависит от их реальной частоты.

In [ ]:
for name, (natural_pred, equal_pred) in strat_predictions.items():
    print("\n", name, "— полная validation")
    print(classification_report(strat_validation.Severity, natural_pred, labels=LABELS, zero_division=0))
    print("Равные классы validation")
    print(classification_report(equal_validation.Severity, equal_pred, labels=LABELS, zero_division=0))

## 22. Можно ли улучшить модель: Gradient Boosting

Попробуем ещё одну модель из scikit-learn: **HistGradientBoostingClassifier**. Она строит много небольших деревьев и может находить более сложные зависимости между признаками. Используем координаты начала ДТП, месяц и признаки дорожной инфраструктуры вместе с прежними признаками.

Берём те же 30% данных и то же случайное стратифицированное разбиение, что в разделе 21. В обучении делаем по 50 000 строк на каждый класс: редкие записи повторяются, поэтому уникальных записей меньше 200 000. Преобразования обучаются только на train. Результаты сравниваем на двух одинаковых validation-выборках из раздела 21: полной и с равными классами.

**Как понимать результат:** равная validation помогает сравнить классы при одинаковой частоте. Полная validation показывает результат при исходных частотах данных. Это разные условия, поэтому выводим обе оценки. Случайное разделение не гарантирует такого же качества на данных будущих лет; время доступности координат и дорожных признаков в реальной системе также нужно отдельно подтвердить. Эти эксперименты используют validation для выбора; test не оцениваем.

In [ ]:
"""Проверка градиентного бустинга на сбалансированном train."""
import time
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from threadpoolctl import threadpool_limits


def create_boosting():
    """Признаки готовятся только на train и используются бустингом."""
    numeric = ['hour', 'day_of_week'] + WEATHER_NUMBERS + EXTRA_NUMERIC
    category = ['State', 'Sunrise_Sunset'] + WEATHER_CATEGORY + EXTRA_CATEGORIES
    prep = ColumnTransformer([
        ('numbers', SimpleImputer(strategy='median', keep_empty_features=True), numeric),
        ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), category),
    ], sparse_threshold=0)
    classifier = HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=31,
                                                 learning_rate=0.1, random_state=42,
                                                 early_stopping=False)
    return Pipeline([('prepare', prep), ('model', classifier)])


def evaluate_boosting(train, validation):
    """Метрики на validation с естественным и равным распределением."""
    balanced_train = balance_train(train, target_per_class=50000)
    balanced_val = balanced_evaluation(validation)
    model = create_boosting()
    print('Число строк balanced train:', len(balanced_train), flush=True)
    print('Число уникальных ID:', balanced_train.ID.nunique(), flush=True)
    print('Обучаем Gradient Boosting...', flush=True)
    with threadpool_limits(limits=4):
        seconds, memory = fit_with_measurement(model, balanced_train[EXTENDED_FEATURES], balanced_train.Severity)
        natural_pred = model.predict(validation[EXTENDED_FEATURES])
        balanced_pred = model.predict(balanced_val[EXTENDED_FEATURES])
    scores = {'model':'Gradient Boosting + road / location',
              'train_rows':len(balanced_train),
              'accuracy_natural':accuracy_score(validation.Severity, natural_pred),
              'f1_macro_natural':f1_score(validation.Severity, natural_pred, labels=LABELS, average='macro', zero_division=0),
              'accuracy_equal_classes':accuracy_score(balanced_val.Severity, balanced_pred),
              'f1_macro_equal_classes':f1_score(balanced_val.Severity, balanced_pred, labels=LABELS, average='macro', zero_division=0),
              'train_seconds':seconds, 'peak_process_ram_mib':memory}
    return scores, model, natural_pred, balanced_val, balanced_pred


In [ ]:
# Нужные столбцы читаются сразу — без соединения всей выборки второй раз.
extra_fields = ['Start_Lat', 'Start_Lng'] + EXTRA_CATEGORIES
boost_sample = create_model_sample(OUTPUT_DIR, sample_percent=30, extra_columns=extra_fields)
boost_sample['month'] = boost_sample['start'].dt.month
for name in ['Start_Lat', 'Start_Lng']:
    boost_sample[name] = pd.to_numeric(boost_sample[name], errors='coerce')
for name in EXTRA_CATEGORIES:
    boost_sample[name] = boost_sample[name].fillna('Unknown').astype(str)

boost_train, boost_validation, boost_test = stratified_split(boost_sample)
print('Train:', len(boost_train), '| validation:', len(boost_validation), '| test:', len(boost_test))
# Целевая колонка test не используется в эксперименте.
boost_scores, boost_model, boost_pred, boost_equal_val, boost_equal_pred = evaluate_boosting(boost_train, boost_validation)
display(pd.DataFrame([boost_scores]).round(4))
pd.DataFrame([boost_scores]).to_csv('/content/traffic_project/boosting_validation.csv', index=False)

In [ ]:
print('Полная validation с исходными долями:')
print(classification_report(boost_validation['Severity'], boost_pred, labels=LABELS, zero_division=0))
print('Validation с равным числом записей каждого класса:')
print(classification_report(boost_equal_val['Severity'], boost_equal_pred, labels=LABELS, zero_division=0))